# Do predictions hold for genes nobody has studied?


**The question.** A model can look good on average by being right about well-studied genes. Here the frozen test of the glucose question is read again within three strata of study effort -- no publication, 1-4, 5 or more -- with the threshold and calibration chosen on train + tune applied unchanged (instruction 64.16). Publication counts are attention and are held out as evidence; they are used here only to split the test genes.

Built 2026-11-07.

In [1]:
import os, numpy as np, pandas as pd
from parascape import harness as H, question_registry as Q, datasets as ds, organisms as O
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet')).set_index('gene_id')
q = Q.get('required_without_glucose')
y = (nodes['fit_no_glucose'] <= -2.0)[nodes['fit_no_glucose'].notna()]
def family_of(c):
    f = ds.provenance(c, O.TOXOPLASMA); return f.key if f is not None else 'unregistered'
numeric = [c for c in nodes.columns if pd.api.types.is_numeric_dtype(nodes[c]) or pd.api.types.is_bool_dtype(nodes[c])]
families = pd.Series({c: family_of(c) for c in numeric})
X = H.allowed_features(q, nodes.loc[y.index, numeric].astype('float64'), families)
groups = nodes['orthogroup'].reindex(y.index)
groups = groups.where(groups.notna(), pd.Series({g: 'self:' + g for g in y.index}))
pubs = nodes.loc[y.index, 'n_publications']
strata = pd.cut(pubs, [-1, 0, 4, np.inf], labels=['no publication', '1-4 publications', '5 or more'])
strata.value_counts()

n_publications
no publication      6670
1-4 publications     590
5 or more            133

In [2]:
r = H.run_question(q, labels=y, features=X, families=families, groups=groups,
                   strategies=[H.Strategy.of('boosted trees (reference)', 'hist_gbt', max_iter=150)],
                   seed=20, importance_repeats=1, strata=strata.astype(str),
                   log_path=os.path.join(REPO, H.HOLDOUT_LOG))
table = pd.DataFrame(r.frozen_test['by_stratum']).T
table.loc['all'] = {k: r.frozen_test[k] for k in table.columns if k in r.frozen_test}
table.to_csv(os.path.join(OUT, 'by_stratum.tsv'), sep='\t')
table

,n,positives,calls,precision,precision_lower90,roc_auc,average_precision,calibration_error
1-4 publications,87,26,4,0.75,0.3205,0.8531,0.6814,0.07961
5 or more,19,5,0,NaN,NaN,0.8571,0.75,0.1336
no publication,1027,223,18,0.7778,0.604,0.8313,0.5386,0.0163
all,1133,254,22,0.7727,0.6188,0.8337,0.5549,0.01611
